# 03 · Small models + robustness data (E1, E4, E6, E7 data)

**Runtime:** T4 GPU (High-RAM not needed). **Run after:** 01. **Run before:** 05 (the A100 notebook needs the robustness files). **Time:** ~1.5–2.5 h on T4.

1. BiLSTM baseline (E1) and BiLSTM claim-only (E4), 3 seeds each
2. Zero-shot `roberta-large-mnli` (E1), with gold and retrieved evidence
3. E6 test sets: distractor evidence and back-translated (en→de→en) claims, filtered by mutual entailment
4. E7 training paraphrases: 15K training claims, same filter
5. `handcheck_paraphrases.csv` in Drive: 100 random paraphrases to label by hand

Every step is skipped if its output already exists, so after a disconnect just run all again.

In [ ]:
# Setup: Drive, code, packages, paths  (expected runtime: T4)
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = '/content/drive/MyDrive/neural-evidence'
CODE = '/content/drive/MyDrive/Colab Notebooks/neural-evidence'   # your uploaded code folder
REPO_URL = ''
if not os.path.exists(f'{CODE}/ne') and REPO_URL:
    os.system(f'git clone -q {REPO_URL} /content/code'); CODE = '/content/code'
assert os.path.exists(f'{CODE}/ne'), f'Code not found at {CODE}'
os.chdir(CODE); sys.path.insert(0, CODE)
!pip -q install -r requirements.txt 
PROC, RESULTS, CACHE = f'{BASE}/data/processed', f'{BASE}/results', f'{BASE}/cache'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
from ne import runtime
from ne.data import read_jsonl, write_jsonl, eval_sets
runtime.setup()
if runtime.gpu_kind() not in 'T4'.split('/') and 'T4' != 'CPU':
    print('WARNING: expected T4; this runtime is', runtime.gpu_kind(), '- it will still work, just at a different speed/cost')
EVALS = eval_sets(PROC); print('eval sets:', list(EVALS))

In [ ]:
from ne import bilstm, results, zeroshot, perturb
import numpy as np, random, torch
train, val = read_jsonl(f'{PROC}/fever/train.jsonl'), read_jsonl(f'{PROC}/fever/val.jsonl')
tests = {k: read_jsonl(EVALS[k]) for k in ('fever_test', 'scifact_test')}
labels = {k: [r['label'] for r in v] for k, v in tests.items()}

## 1 · BiLSTM (E1, E4 claim-only)

In [ ]:
glove = bilstm.download_glove(f'{CACHE}/glove')
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for exp, name, ev in [('E1', 'bilstm', 'gold'), ('E4', 'bilstm_claim_only', 'none')]:
    for seed in (42, 43, 44):
        if results.is_done(RESULTS, exp, name, seed):
            print('skip', name, seed); continue
        model, vocab, info = bilstm.train(train, val, glove, evidence=ev, seed=seed)
        preds = {k: bilstm.predict(model, v, vocab, ev, dev) for k, v in tests.items()}
        results.save_run(RESULTS, exp, name, seed, {'evidence': ev, **info}, preds, labels,
                         extra={'train_seconds': info['train_seconds'],
                                'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 2**30, 2) if dev.type == 'cuda' else 0},
                         rows_by_set=tests)
        del model; torch.cuda.empty_cache()

## 2 · Zero-shot MNLI (E1)
MNLI-only checkpoint on purpose: models trained on FEVER would leak the test set.

In [ ]:
scorer = zeroshot.NLIScorer('roberta-large-mnli')
for name, field in [('zeroshot_mnli', 'evidence_gold'), ('zeroshot_mnli_retrieved', 'evidence_retrieved')]:
    if results.is_done(RESULTS, 'E1', name, 0):
        print('skip', name); continue
    preds, secs = {}, 0
    for k, v in tests.items():
        preds[k], s = zeroshot.predict_rows(scorer, v, field); secs += s
    results.save_run(RESULTS, 'E1', name, 0, {'model': 'roberta-large-mnli', 'evidence_field': field},
                     preds, labels, extra={'inference_seconds': secs}, rows_by_set=tests)

## 3 · E6 robustness test sets

In [ ]:
ROB = f'{PROC}/robustness'; os.makedirs(ROB, exist_ok=True)
fever_test = tests['fever_test']
if not os.path.exists(f'{ROB}/fever_test_distract.jsonl'):
    write_jsonl(perturb.add_distractors(fever_test, n_sents=2, seed=0), f'{ROB}/fever_test_distract.jsonl')
print('distractor set ready')

bt = None
def back_translate_rows(rows, path_all):
    global bt
    if os.path.exists(path_all):
        return read_jsonl(path_all)
    bt = bt or perturb.BackTranslator('de')
    claims = [r['claim'] for r in rows]
    para = bt.paraphrase(claims)
    score = perturb.mutual_entailment(scorer, claims, para)
    out = perturb.paraphrase_rows(rows, para, score, threshold=0.5)
    write_jsonl(out, path_all)
    return out

para_all = back_translate_rows(fever_test, f'{ROB}/fever_test_para_all.jsonl')
keep = [r for r in para_all if r['para_changed'] and r['para_nli_ok']]
write_jsonl(keep, f'{ROB}/fever_test_para.jsonl')
# the matching original claims, so E6 compares like with like
ids = {r['claim_original'] for r in keep}
write_jsonl([r for r in fever_test if r['claim'] in ids], f'{ROB}/fever_test_para_orig.jsonl')
print(f'paraphrased test: {len(keep)}/{len(para_all)} kept (changed and mutual entailment >= 0.5)')

## 4 · E7 training paraphrases (15K claims from the training split)

In [ ]:
rng = random.Random(0)
src = [train[i] for i in sorted(rng.sample(range(len(train)), 15000))]
tr_all = back_translate_rows(src, f'{ROB}/train_para_15k_all.jsonl')
tr_keep = [r for r in tr_all if r['para_changed'] and r['para_nli_ok']]
write_jsonl(tr_keep, f'{ROB}/train_para_15k.jsonl')
print(f'training paraphrases kept: {len(tr_keep)}/{len(tr_all)}')

In [ ]:
# 100 random paraphrases (50 test, 50 train) for the hand check; fill the 'label_preserved' column (y/n)
import pandas as pd
r = random.Random(1)
sample = r.sample(keep, 50) + r.sample(tr_keep, 50)
pd.DataFrame([{'set': 'test' if i < 50 else 'train', 'label': s['label_name'], 'original': s['claim_original'],
               'paraphrase': s['claim'], 'nli_score': round(s['para_nli_entail'], 3), 'label_preserved': ''}
              for i, s in enumerate(sample)]).to_csv(f'{BASE}/handcheck_paraphrases.csv', index=False)
print('saved', f'{BASE}/handcheck_paraphrases.csv')